In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Navigate to your Drive
%cd /content/drive/MyDrive/iambesideyou

# Create project folders
!mkdir intern_task_automation
%cd intern_task_automation
!mkdir src

# Initialize Git
!git init

# Set your Git identity
!git config --global user.email "aryan.srivastava@iitg.ac.in"
!git config --global user.name "Aryan"

/content/drive/MyDrive/iambesideyou
/content/drive/MyDrive/iambesideyou/intern_task_automation
hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/drive/MyDrive/iambesideyou/intern_task_automation/.git/


# DAY 1


In [5]:
import pandas as pd
import json
import glob

# 1. Automatically search for the dataset_a folder in your Drive
# This checks both your root MyDrive and inside your iambesideyou folder
search_paths = [
    '/content/drive/MyDrive/data/dataset_a/ses_*/chunk_*/events.jsonl',
    '/content/drive/MyDrive/iambesideyou/data/dataset_a/ses_*/chunk_*/events.jsonl'
]

all_event_files = []
for path in search_paths:
    all_event_files.extend(glob.glob(path))

if all_event_files:
    # Pick the first available events.jsonl file
    chunk_path = all_event_files[0]
    print(f"✅ Automatically found chunk path: {chunk_path}\n")

    # 2. Define the loading function
    def load_events(file_path):
        events = []
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                events.append(json.loads(line))

        # Flatten the nested JSON (payload, context, etc.)
        df = pd.json_normalize(events)
        df['timestamp_iso'] = pd.to_datetime(df['timestamp_iso'])
        df = df.sort_values('timestamp_iso').reset_index(drop=True)
        return df

    # 3. Load and display the data
    df_events = load_events(chunk_path)
    print(f"Loaded {len(df_events)} events.")
    display(df_events.head())
else:
    print("❌ Could not find the events.jsonl files. Double-check that the 'AI Engineer' shortcut was added to your Drive.")

✅ Automatically found chunk path: /content/drive/MyDrive/iambesideyou/data/dataset_a/ses_20260630-121953-LAPTOP-R36BQBTE/chunk_20260630-1200-LAPTOP-R36BQBTE/events.jsonl

Loaded 1116 events.


,schema_version,event_id,session_id,timestamp_ms,timestamp_iso,layer,event_type,source.agent_version,source.machine_id,source.os,...,payload.input_context.is_readonly,extensions.uia_v2.nearby_context.container_name_hash,payload.new_state,payload.previous_state,payload.window_title,payload.corrections_count,payload.duration_ms,payload.final_text,payload.keystroke_count,payload.related_keystrokes
0,1.0.0,evt_f29ff271-1691-4070-9b8f-02aeead78ff2,ses_20260630-121953-LAPTOP-R36BQBTE,1782821993821,2026-06-30 12:19:53.821000+00:00,SYSTEM,session_start,1.1.1,LAPTOP-R36BQBTE,Windows Windows_NT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.0.0,evt_dfd59008-2c05-441c-b31e-3a536c8eb188,ses_20260630-121953-LAPTOP-R36BQBTE,1782821994435,2026-06-30 12:19:54.435000+00:00,L2,mouse_click,1.1.1,LAPTOP-R36BQBTE,Windows Windows_NT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.0.0,evt_0ffceab0-84e5-41c1-8a83-ad9504214bee,ses_20260630-121953-LAPTOP-R36BQBTE,1782821994465,2026-06-30 12:19:54.465000+00:00,L2,app_switch,1.1.1,LAPTOP-R36BQBTE,Windows Windows_NT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.0.0,evt_84a34d4d-53c8-4123-bde7-bb8f3e3fae9a,ses_20260630-121953-LAPTOP-R36BQBTE,1782821994524,2026-06-30 12:19:54.524000+00:00,L1,screenshot_smart,1.1.1,LAPTOP-R36BQBTE,Windows Windows_NT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.0.0,evt_bffc7e35-dcba-4d03-9372-5cf62f5a4f36,ses_20260630-121953-LAPTOP-R36BQBTE,1782821995403,2026-06-30 12:19:55.403000+00:00,L2,keystroke,1.1.1,LAPTOP-R36BQBTE,Windows Windows_NT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
%cd /content/drive/MyDrive/iambesideyou/intern_task_automation
!git add .
!git commit -m "Day 1: Initialized repo, mounted drive, and loaded events.jsonl"

/content/drive/MyDrive/iambesideyou/intern_task_automation
On branch master
nothing to commit, working tree clean


In [10]:
import os
import pandas as pd
import json

# Navigate one level up from the chunk folder to get the session folder
session_dir = os.path.dirname(os.path.dirname(chunk_path))
gt_path = os.path.join(session_dir, 'gt.jsonl')

print(f"Loading ground truth from: {gt_path}\n")

gt_events = []
with open(gt_path, 'r', encoding='utf-8') as f:
    for line in f:
        gt_events.append(json.loads(line))

df_gt = pd.DataFrame(gt_events)

# Convert UTC timestamps to pandas datetime objects for easier alignment
if 'ts_utc' in df_gt.columns:
    df_gt['ts_utc'] = pd.to_datetime(df_gt['ts_utc'])

print(f"Loaded {len(df_gt)} ground truth boundaries.")
display(df_gt.head())

Loading ground truth from: /content/drive/MyDrive/iambesideyou/data/dataset_a/ses_20260630-121953-LAPTOP-R36BQBTE/gt.jsonl

Loaded 175 ground truth boundaries.


,ts_utc,run_id,event,seed,dwell_scale,n_procs,run_date,operator,operator_dept,machine_id,...,note_id,target_app,target_field,from,to,process_variant,split_id,phase,total_tasks,duration_seconds
0,2026-06-30 12:20:25.140811+00:00,theme_m1_20260630_175009,run_config,645230.0,1.4,9.0,2026-06-30,田中 健一,人事部,NB-M1-05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-06-30 12:21:12.794418+00:00,theme_m1_20260630_175009,process_started,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-06-30 12:21:12.797484+00:00,theme_m1_20260630_175009,task_started,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-06-30 12:21:23.693988+00:00,theme_m1_20260630_175009,clipboard_copy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,BR-175009-001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-06-30 12:21:23.696049+00:00,theme_m1_20260630_175009,clipboard_paste,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,BR-175009-001,excel,find,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
# Filter for meaningful events like app switches and keystrokes
df_features = df_events[df_events['event_type'].isin(['app_switch', 'keystroke', 'mouse_click'])].copy()

# json_normalize already flattened the nested data into these column names
target_columns = [
    'timestamp_iso',
    'event_type',
    'context.active_app.app_name',
    'context.active_app.window_title',
    'correlation.ms_since_last_event'
]

# Only select columns that actually exist in this specific chunk to avoid future KeyErrors
available_cols = [col for col in target_columns if col in df_features.columns]

print("Extracted Features:")
display(df_features[available_cols].head(10))

Extracted Features:


,timestamp_iso,event_type,context.active_app.app_name,context.active_app.window_title,correlation.ms_since_last_event
1,2026-06-30 12:19:54.435000+00:00,mouse_click,WindowsTerminal,Theme M1 - Mixed Multi-Domain (HR / Finance / ...,0.0
2,2026-06-30 12:19:54.465000+00:00,app_switch,WindowsTerminal,Theme M1 - Mixed Multi-Domain (HR / Finance / ...,5.0
4,2026-06-30 12:19:55.403000+00:00,keystroke,WindowsTerminal,Theme M1 - Mixed Multi-Domain (HR / Finance / ...,884.0
6,2026-06-30 12:19:55.514000+00:00,keystroke,WindowsTerminal,Theme M1 - Mixed Multi-Domain (HR / Finance / ...,111.0
7,2026-06-30 12:19:57.385000+00:00,keystroke,WindowsTerminal,Theme M1 - Mixed Multi-Domain (HR / Finance / ...,1919.0
9,2026-06-30 12:19:57.520000+00:00,keystroke,WindowsTerminal,Theme M1 - Mixed Multi-Domain (HR / Finance / ...,139.0
10,2026-06-30 12:20:00.044000+00:00,keystroke,WindowsTerminal,Theme M1 - Mixed Multi-Domain (HR / Finance / ...,2635.0
12,2026-06-30 12:20:00.174000+00:00,keystroke,WindowsTerminal,Theme M1 - Mixed Multi-Domain (HR / Finance / ...,133.0
13,2026-06-30 12:20:02.290000+00:00,keystroke,WindowsTerminal,Theme M1 - Mixed Multi-Domain (HR / Finance / ...,2136.0
15,2026-06-30 12:20:02.425000+00:00,keystroke,WindowsTerminal,Theme M1 - Mixed Multi-Domain (HR / Finance / ...,138.0
